## Cell 1: Setup, Imports, Tools, and API Integration
In this cell, I set up the foundation of my multi-agent system.

I initialized Python's logging module to enable Observability, a practice learned in Day 4, which helps track agent flow and key decisions.

For model interaction, I securely load the 'GOOGLE_API_KEY' and define the llm_call function, which simulates the core reasoning of an LlmAgent from Day 1.

I also define my Custom Function Tools *(Day 2)*:


* knowledge_fetcher_tool: Simulates a RAG lookup to ground the TutorAgent.
* question_generator_tool: Generates structured data (MCQs), ensuring reliable output for the assessment phase.


In [25]:
import json
import os
import random
from typing import List, Dict, Any, Tuple
import time 
import logging

# --- Configure Logging (Observability Bonus) ---
# I configure basic logging to track agent flow and key decisions.
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# --- SECRETS LOAD & GEMINI CLIENT ---
# This is how I load the API key for secure authentication.
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

try:
    GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    logging.info("GOOGLE_API_KEY loaded and set in environment.")
except Exception:
    logging.warning("GOOGLE_API_KEY not found. Using dummy LLM function.")

# --- GEMINI CLIENT & CONFIGURATION (Using actual API) ---
try:
    from google import genai
    from google.genai import types
    gemini_client = genai.Client()
    MODEL_NAME = "gemini-2.5-flash"
    
    def llm_call(prompt: str, role: str) -> str:
        """My actual Gemini API caller function."""
        if role == "tutor":
            system_instruction = "You are a specialized TutorAgent. Provide clear, step-by-step explanations for the topic."
        elif role == "planner":
            system_instruction = "You are a PlannerAgent. Analyze the provided memory data and generate a concise, actionable 7-day revision plan focused on weaknesses."
        else:
            system_instruction = "You are a helpful assistant."

        config = types.GenerateContentConfig(
            system_instruction=system_instruction
        )

        try:
            response = gemini_client.models.generate_content(
                model=MODEL_NAME,
                contents=[prompt],
                config=config
            )
            logging.info(f"LLM Call successful for {role}")
            return response.text
        except Exception as e:
            logging.error(f"LLM API Error for {role}: {e}")
            return f"LLM API Error for {role}: Using fallback dummy."

except ImportError:
    # Fallback to dummy function if SDK or API setup fails
    def llm_call(prompt: str, role: str) -> str:
        if role == "tutor":
            return f"Tutor Explanation (Dummy): The topic is simplified as: {prompt}. Proceeding to quiz."
        elif role == "planner":
            if "Energy Source" in prompt:
                return "--- Personalized 7-Day Revision Plan (Dummy) --- Day 1: Focus on Energy Source."
            return "--- Generic Revision Plan (Dummy) ---"
        return "LLM Response (Dummy)."


# --- CUSTOM TOOLS (Function Tools) ---
def knowledge_fetcher_tool(topic: str) -> str:
    """Retrieves curated information. (Function Tool)"""
    if "Photosynthesis" in topic:
        return (
            "Photosynthesis is the process used by plants to convert light energy, usually from the Sun, into chemical energy "
            "stored in sugars, using carbon dioxide and water. Key stages: Light-Dependent Reactions and Light-Independent Reactions (Calvin Cycle)."
        )
    return f"Information snippet for {topic}: [Curated knowledge for the topic]."

def question_generator_tool(explanation: str) -> Tuple[List[Dict[str, str]], List[str]]:
    """Generates structured MCQs and identifies sub-topics. (Function Tool)"""
    if "Photosynthesis" in explanation:
        mcqs = [
            {"q": "What is the primary energy source for photosynthesis?", "options": ["Water", "Light", "Soil"], "answer": "Light"},
            {"q": "Which reaction converts light energy into chemical energy?", "options": ["Calvin Cycle", "Light-Dependent", "Respiration"], "answer": "Light-Dependent"},
        ]
        weak_topics = ["Light-Dependent Reactions", "Energy Source"] 
        return mcqs, weak_topics
    return [{"q": "Generic Q?", "options": ["A", "B", "C"], "answer": "A"}], ["Generic Weakness"]

INFO: GOOGLE_API_KEY loaded and set in environment.


## Cell 2: The Memory System Class (Day 3)
This cell defines the MemoryStore class, which simulates the necessary long-term persistence and retrieval provided by a dedicated MemoryService.

* Persistence Across Sessions (Day 3)

The MemoryStore fulfills the role of long-term memory, retaining student performance data even after a single learning Session has ended. This persistence is foundational to building an adaptive and personalized tutor. The initialization logic handles loading existing data for persistence or resetting for a new demo.

### Memory Ingestion and Retrieval

* update_quiz_result: This method is the core of Memory Ingestion (analogous to add_session_to_memory). It is called by the QuizAgent to record the latest performance data, specifically tracking and aggregating weak topics in the weak_topics dictionary.

* get_full_history: This method simulates Memory Retrieval (akin to load_memory). It ensures the PlannerAgent can access all historical weakness data required to generate an adaptive plan. The logging.info call here provides an Observability trace that the PlannerAgent is retrieving the necessary memory.

In [26]:
# --- MEMORY SYSTEM (Simulating MemoryService) ---
class MemoryStore:
    def __init__(self, filename='student_memory.json'):
        self.filename = filename
        self.memory = {}
        self._initialize_memory()

    def _initialize_memory(self):
        # Load existing memory or initialize new structure to ensure persistence.
        if os.path.exists(self.filename):
            try:
                with open(self.filename, 'r') as f:
                    self.memory = json.load(f)
            except json.JSONDecodeError:
                self.reset_memory()
        else:
            self.reset_memory()

    def reset_memory(self):
        # I reset the structure for a clean demo run.
        self.memory = {
            "quiz_history": [],
            "weak_topics": {}, 
            "tutor_notes": ""
        }
        self.save_memory()
        
    def save_memory(self):
        # Persists the state.
        with open(self.filename, 'w') as f:
            json.dump(self.memory, f, indent=4)

    def update_quiz_result(self, topic: str, score: int, weaknesses: List[str]):
        """Records quiz outcome and updates the count for identified weak topics (simulating memory ingestion)."""
        self.memory["quiz_history"].append({
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "topic": topic,
            "score_percentage": score,
        })
        # I update the long-term weakness tracking here.
        for weak_topic in weaknesses:
            self.memory["weak_topics"][weak_topic] = self.memory["weak_topics"].get(weak_topic, 0) + 1
        self.save_memory()
        logging.info(f"Memory update: Score {score}% recorded for {topic}. Weaknesses updated.")
        
    def get_full_history(self) -> Dict[str, Any]:
        """Retrieves all memory data (simulating memory search/retrieval for the PlannerAgent)."""
        logging.info("Memory access: Full history retrieved for analysis.")
        return self.memory

## Cell 3: Agent Definitions (Day 1 & Day 2)
This cell defines the specialized logic for my three core LlmAgents (Tutor, Quiz, Planner) that form my multi-agent team.

### TutorAgent : The Explainer

The TutorAgent acts as the first stage of the workflow.

Tool Usage (Day 2): It calls the knowledge_fetcher_tool to retrieve content before engaging the LLM to explain the topic. This ensures the explanation is grounded in the correct data (RAG).

Logging (Day 4): The logging.info call confirms the agent was invoked, which aids in tracing the start of the overall sequence.

Memory Action (Day 3): It records initial tutor_notes to the shared MemoryStore to persist context for future sessions.

### QuizAgent χ: The Assessor

The QuizAgent handles the assessment and data capture stage.

Tool Usage (Day 2): It utilizes the question_generator_tool to produce structured quiz content based on the lesson's explanation, showing specialized tool use.

Memory Ingestion (Day 3): It calls update_quiz_result, simulating Memory Ingestion (like add_session_to_memory) to record the score and identified weak topics.

Loop Logic (Day 1 - Bonus): The if score < 75 statement is a decision point, which controls the logic of the external LoopAgent pattern in the orchestrator, determining whether to trigger Iterative Refinement.

### PlannerAgent : The Adaptive Intelligence

The PlannerAgent executes the system's core adaptive function.

Memory Retrieval (Day 3): The logging.info call confirms the agent performs Memory Retrieval via get_full_history. This ensures the planning process is grounded in the student's historical performance data, not just the current session.

Final Output: It uses the LLM to analyze the retrieved data, generating the customized 7-day plan, which is the final output of the sequential workflow.

In [27]:
# --- AGENT DEFINITIONS (LlmAgent equivalents) ---

class TutorAgent:
    """Explains concepts and saves initial notes to the shared MemoryStore."""
    def __init__(self, memory: MemoryStore):
        self.memory = memory
    
    def teach_topic(self, topic: str) -> Tuple[str, str]:
        logging.info("TutorAgent: Invoked to fetch and teach content.")
        knowledge = knowledge_fetcher_tool(topic)
        
        prompt = f"Use this knowledge to explain the topic: {knowledge}"
        explanation = llm_call(prompt, role="tutor")
        
        self.memory.memory["tutor_notes"] = explanation[:50] + "..."
        self.memory.save_memory()
        
        return explanation, knowledge

class QuizAgent:
    """Generates MCQs, simulates student answers, grades the quiz, and updates MemoryStore."""
    def __init__(self, memory: MemoryStore):
        self.memory = memory
    
    def run_quiz(self, topic: str, explanation: str):
        logging.info("QuizAgent: Running assessment and grading.")
        mcqs, weak_topics_list = question_generator_tool(explanation)
        
        print(f"\n--- Quiz Time on {topic}! ---")
        
        correct_answers = 0
        
        # Simulate student performance for this attempt.
        for i, q in enumerate(mcqs):
            # 50% chance of getting the answer correct for demo purposes
            is_correct = random.choice([True, False]) 
            if is_correct:
                 correct_answers += 1
        
        score = int((correct_answers / len(mcqs)) * 100) if mcqs else 0
        
        # Log decision point: was the score below the target?
        if score < 75:
             logging.warning(f"QuizAgent: Score {score}% is below target. Will loop for revision.")
        else:
             logging.info(f"QuizAgent: Target score ({score}%) achieved. Exiting loop.")
             
        self.memory.update_quiz_result(topic, score, weak_topics_list)
        
        return score, weak_topics_list

class PlannerAgent:
    """Reads memory for performance data and generates a personalized revision plan."""
    def __init__(self, memory: MemoryStore):
        self.memory = memory
    
    def generate_plan(self):
        logging.info("PlannerAgent: Retrieving data from MemoryStore.")
        memory_data = self.memory.get_full_history()
        
        prompt = f"Analyze this student memory data and create a 7-day revision plan, focusing on weak topics. Data: {memory_data}"
        plan = llm_call(prompt, role="planner")
        
        return plan

## Cell 4: Orchestrator (Sequential + LoopAgent Logic)  (Day 1)
This cell defines the Orchestrator class, which enforces the complex workflow logic of the tutoring system by combining two core agent architectures learned in Day 1.

### Sequential Workflow Pattern (Day 1)

The entire run_learning_cycle method implements a Sequential Agent pattern. The steps are linear: TutorAgent → Quiz Loop → PlannerAgent, ensuring data is generated, assessed, and then analyzed in the correct order.

### LoopAgent Complexity (Bonus Point)

The central while loop simulates the LoopAgent pattern, providing Iterative Refinement.

Condition: The loop continues as long as current_score is below the TARGET_SCORE (75%).

Micro-Planning: After each failed attempt, the Orchestrator immediately calls the PlannerAgent to generate a "micro-plan." This demonstrates using the Memory system immediately after ingestion to provide personalized feedback during the revision loop.

Observability (Day 4): The logging.info calls trace the execution path and log the starting and ending points of the primary agents, confirming the workflow and debugging steps.

In [28]:
class Orchestrator:
    """The central coordinator that defines the sequential flow (Tutor -> Quiz Loop -> Planner)."""
    TARGET_SCORE = 75
    MAX_ATTEMPTS = 3

    def __init__(self):
        # I initialize the core ADK components: MemoryStore and my three specialized agents.
        self.memory = MemoryStore()
        self.tutor_agent = TutorAgent(self.memory)
        self.quiz_agent = QuizAgent(self.memory)
        self.planner_agent = PlannerAgent(self.memory)

    def run_learning_cycle(self, topic: str):
        """Executes the full learning path with a revision loop (simulating LoopAgent)."""
        print(f"\n" + "="*70)
        print(f"### 🚀 Starting Personalized Learning Cycle for: **{topic}** ###")
        print("="*70)
        
        # 1. TutorAgent: Teach (Sequential Step 1)
        logging.info("Orchestrator: Starting TutorAgent.")
        explanation, _ = self.tutor_agent.teach_topic(topic)
        print(f"Tutor Response: {explanation}")
        
        current_score = 0
        attempt = 0

        # 2. Quiz and Revision Loop (Simulating LoopAgent)
        while current_score < self.TARGET_SCORE and attempt < self.MAX_ATTEMPTS:
            attempt += 1
            print(f"\n[2/3. Quiz Attempt #{attempt}: Assessing Knowledge...]")
            
            # The QuizAgent runs here.
            score, weak_topics = self.quiz_agent.run_quiz(topic, explanation)
            current_score = score
            
            print(f"Memory Check: Score {score}% recorded.")
            
            if current_score < self.TARGET_SCORE:
                print(f"--- Score below target ({self.TARGET_SCORE}%). Simulating revision period. ---")
                
                # I use the PlannerAgent here to generate a micro-plan after failure, 
                # demonstrating the benefit of the memory system immediately.
                if attempt < self.MAX_ATTEMPTS:
                    print("PlannerAgent Micro-Plan:")
                    print(self.planner_agent.generate_plan())
                
        # End of Loop logic (Equivalent to LoopAgent exit condition)
        print("\n--- Revision Loop Concluded ---")
        if current_score >= self.TARGET_SCORE:
            print(f"🎉 SUCCESS: Target score ({self.TARGET_SCORE}%) achieved after {attempt} attempts!")
        else:
            print(f"⚠️ Max attempts reached. Final score: {current_score}%. Proceeding to long-term plan.")


        # 3. PlannerAgent: Final Long-Term Plan (Sequential Step 3)
        logging.info("Orchestrator: Generating final long-term PlannerAgent output.")
        print("\n[3/3. Planner Agent: Generating Final Personalized Plan...]")
        revision_plan = self.planner_agent.generate_plan()
        print(f"Planner Response:\n{revision_plan}")
        
        print("\n### ✅ Learning Cycle Complete. ###")
        print(f"Weak Topics Logged in Memory:\n{self.memory.get_full_history()['weak_topics']}")
        print("="*70)

# Now Comes the Fun Part,  The Demo...

## Cell 5: Demo Execution and Adaptation Test
This cell runs the complete, enhanced workflow twice to demonstrate the LoopAgent complexity (Bonus Point) and the final Memory Adaptation capability (Core Requirement).

### Reproducible Demonstration (Day 1, Day 3)

#### The code executes the full Orchestrator cycle, which combines the Sequential Agent workflow (Teach → Loop → Plan) with the LoopAgent logic for iterative refinement.

First Run (Baseline): The system establishes the student's initial performance. Due to the inherent randomness (simulated lack of knowledge), the LoopAgent logic triggers, forcing multiple quiz attempts and generating small Micro-Plans. This process records persistent weak topics into the MemoryStore.

Second Run (Adaptation): I set a fixed random.seed(42) to simulate consistent failure on specific concepts. Crucially, when the PlannerAgent runs in this cycle, it performs Memory Retrieval and generates a personalized revision plan based on the weaknesses logged in the first run, proving the adaptability of the system.

### Deployment Write-Up (Bonus Point) (Day 5)

This section includes the structured text for the "Future Improvements" section of the Capstone write-up. It outlines the strategy for deployment, securing 5 Bonus Points by committing to production-ready technologies:

#### Deployment: The plan centers on deploying the system to Cloud Run or Vertex AI Agent Engine.

Memory Migration: It explicitly states the plan to migrate the JSON MemoryStore to Vertex AI Memory Bank for scalable, persistent long-term storage.

In [31]:
# --- DEMO EXECUTION ---

# 1. Initial Setup: I ensure a clean state before starting.
print("--- Initializing system and cleaning memory for fresh run ---")
# I clean up old memory files to ensure the demo is reproducible
if os.path.exists('student_memory.json'):
    os.remove('student_memory.json')
# Configure logging to output the start of the process
logging.info("Starting memory reset.")
ms_reset = MemoryStore()
ms_reset.reset_memory() 
random.seed(None) # Natural randomness for a realistic demo flow.

# 2. First Run: This establishes the student's baseline performance and logs their weaknesses.
tutor_system = Orchestrator()
tutor_system.run_learning_cycle("Photosynthesis in Plants") 

# ---
# 3. Second Run: I use a fixed random seed here (42) to make sure the student fails consistently 
# on the same topics, demonstrating the reliability of the system's tracking.
random.seed(42) 
print("\n\n" + "#"*70)
print("### 🔄 Second Run to Demonstrate Memory Adaptation (PlannerAgent uses history) ###")
print("#"*70)
tutor_system.run_learning_cycle("Photosynthesis in Plants") 

# --- Deployment Plan for Write-up (Non-executable Python Variable) ---
# I'll store the deployment plan text here using triple quotes to prevent the SyntaxError.
# This text is for the final project write-up, not for execution as a command.

deployment_plan_text = """
### Deployment Plan for Cloud Run / Agent Engine (5 Bonus Points)

If I had more time, I would deploy this entire **Sequential Multi-Agent Tutor** as a web service using **Cloud Run** or **Vertex AI Agent Engine**.

* **Architecture:** The **Orchestrator** (Sequential Agent logic) would be wrapped in an ADK `App` and containerized.
* **Infrastructure:** I would use **Cloud Run** for its serverless nature and automatic scaling down to zero. Alternatively, **Agent Engine** offers a fully managed service specialized for ADK agents.
* **Persistent Memory:** The **MemoryStore** (currently JSON file) would be migrated to **Vertex AI Memory Bank** for production-grade semantic search and persistent, scalable storage.
* **Access:** The agent would be exposed via an API endpoint, allowing students to access the tutor through a simple web interface.
"""
logging.info("Deployment plan text loaded successfully for documentation.")
# To view the content of the plan, you can print the variable:
# print("\n--- BEGIN DEPLOYMENT WRITE-UP DRAFT ---")
# print(deployment_plan_text)
# print("--- END DEPLOYMENT WRITE-UP DRAFT ---")

INFO: Starting memory reset.
INFO: Orchestrator: Starting TutorAgent.
INFO: TutorAgent: Invoked to fetch and teach content.
INFO: AFC is enabled with max remote calls: 10.


--- Initializing system and cleaning memory for fresh run ---

### 🚀 Starting Personalized Learning Cycle for: **Photosynthesis in Plants** ###


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO: LLM Call successful for tutor
INFO: QuizAgent: Running assessment and grading.
INFO: Memory update: Score 50% recorded for Photosynthesis in Plants. Weaknesses updated.
INFO: PlannerAgent: Retrieving data from MemoryStore.
INFO: Memory access: Full history retrieved for analysis.
INFO: AFC is enabled with max remote calls: 10.


Tutor Response: Excellent! Let's break down photosynthesis using the knowledge you've provided.

---

### Understanding Photosynthesis: The Plant's Energy Factory

Photosynthesis is a fundamental biological process that powers nearly all life on Earth. It's essentially how plants (and some other organisms) create their own food using light.

Here's a step-by-step explanation:

---

#### Step 1: The Grand Overview - What is Photosynthesis?

*   **Definition:** Photosynthesis is the process used by plants (and some algae and bacteria) to convert **light energy** (usually from the Sun) into **chemical energy**.
*   **The Goal:** This chemical energy is stored in the form of **sugars** (like glucose), which the plant then uses for growth, repair, and other metabolic activities.
*   **The Ingredients:** To do this, plants primarily use two simple molecules: **carbon dioxide** (from the air) and **water** (from the soil).
*   **The Location:** This entire process takes place within specializ

INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO: LLM Call successful for planner
INFO: QuizAgent: Running assessment and grading.
INFO: Memory update: Score 0% recorded for Photosynthesis in Plants. Weaknesses updated.
INFO: PlannerAgent: Retrieving data from MemoryStore.
INFO: Memory access: Full history retrieved for analysis.
INFO: AFC is enabled with max remote calls: 10.


...

Planner Response:
Here's a 7-day revision plan focused on your weak topics:

**Weaknesses Identified:**
*   Light-Dependent Reactions
*   Energy Source (in the context of Photosynthesis)

---

**7-Day Revision Plan:**

*   **Day 1-2: Master "Energy Source"**
    *   Review how light energy is captured by photosynthetic pigments (chlorophyll).
    *   Understand the electromagnetic spectrum and the role of different wavelengths.
    *   Focus on how light energy is initially converted to chemical energy.
    *   *Action:* Read textbook sections, watch videos on light absorption and pigments, draw a diagram of a chloroplast indicating where light is captured.

*   **Day 3-4: Deep Dive into "Light-Dependent Reactions"**
    *   Study the detailed steps: water splitting (photolysis), electron transport chain (Photosystem I & II), proton gradient formation, and ATP synthase.
    *   Understand the products: ATP, NADPH, and O2.
    *   *Action:* Create a flowchart or detailed diagram of the 

INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO: LLM Call successful for tutor
INFO: QuizAgent: Running assessment and grading.
INFO: QuizAgent: Target score (100%) achieved. Exiting loop.
INFO: Memory update: Score 100% recorded for Photosynthesis in Plants. Weaknesses updated.
INFO: Orchestrator: Generating final long-term PlannerAgent output.
INFO: PlannerAgent: Retrieving data from MemoryStore.
INFO: Memory access: Full history retrieved for analysis.
INFO: AFC is enabled with max remote calls: 10.


Tutor Response: Okay, let's break down photosynthesis step-by-step, using the information you provided!

---

### Topic: Photosynthesis Explained

Photosynthesis is a fundamental process that underpins most life on Earth. It's essentially how plants (and some other organisms) create their own food using sunlight.

Here's a step-by-step explanation:

**Step 1: The Overall Goal - Energy Conversion**
*   **What it is:** Photosynthesis is the process plants use to **convert light energy** (mostly from the Sun) into **chemical energy**.
*   **Why it's important:** This chemical energy is stored in **sugars** (like glucose), which the plant then uses as fuel for growth, repair, and all its metabolic activities. It's like a plant's solar-powered food factory!

**Step 2: The Ingredients (Inputs)**
To make these sugars, plants need three main ingredients:
1.  **Light Energy:** The primary energy source, usually from the Sun.
2.  **Carbon Dioxide (CO₂):** A gas absorbed from the atmosphere throu

INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO: LLM Call successful for planner
INFO: Memory access: Full history retrieved for analysis.
INFO: Deployment plan text loaded successfully for documentation.


Planner Response:
**7-Day Revision Plan: Photosynthesis Weaknesses**

**Focus Topics:** Light-Dependent Reactions, Energy Source

*   **Day 1: Foundation - Energy Source.** Review the initial energy input for photosynthesis (sunlight), its properties, and absorption by pigments. Practice identifying the primary energy source.
*   **Day 2: Deep Dive - Energy Source.** Explore how light energy is captured and converted. Focus on chlorophyll and other pigments' roles. Take a short quiz on 'Energy Source' concepts.
*   **Day 3: Introduction - Light-Dependent Reactions.** Understand the location (thylakoids), overall purpose, and main inputs/outputs of this stage.
*   **Day 4: Mechanics - Light-Dependent Reactions.** Delve into the electron transport chain, ATP, and NADPH formation. Practice sequencing steps and identifying key molecules. Take a short quiz on 'Light-Dependent Reactions'.
*   **Day 5: Integration & Application.** Review how the 'Energy Source' directly drives the 'Light-Depe